# Prepare the Data

In [1]:
%pip install scikit-learn numpy pandas pyarrow spacy
!python3 -m spacy download en_core_web_lg
import pickle
import pandas as pd
from custom_transformers import *


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 16.8 MB/s  0:00:24:00:0100:01

[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')


In [2]:
train_set = pd.read_parquet("train_set.parquet")
y = train_set["diseased"]
X = train_set.drop(columns="diseased")
X.info()

<class 'pandas.DataFrame'>
RangeIndex: 2418 entries, 0 to 2417
Data columns (total 37 columns):
 #   Column                                      Non-Null Count  Dtype         
---  ------                                      --------------  -----         
 0   X                                           2418 non-null   float64       
 1   Y                                           2418 non-null   float64       
 2   Unique Squirrel ID                          2418 non-null   str           
 3   Hectare                                     2418 non-null   str           
 4   Shift                                       2418 non-null   str           
 5   Date                                        2418 non-null   datetime64[us]
 6   Hectare Squirrel Number                     2418 non-null   int64         
 7   Age                                         2330 non-null   str           
 8   Primary Fur Color                           2375 non-null   str           
 9   Highlight Fur Color

We fill missing numeric values with the training median and scale the numeric features. For categorical features, we give missing values their own category and use one-hot encoding. We keep weather and park event features in separate groups so we can test during training whether each group helps identify sick sightings.

We use the sighting notes to calculate similarity scores for aggression and play. We leave out the fear score because it was strongly correlated with aggression in the training data (0.93). When the behavior notes are empty, we set both scores to zero and use a separate `has_behavior_text` flag to record whether the sighting had a note.

In [3]:
preprocessor = make_preprocessor()
preprocessor

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('clean', ...), ('notes', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('weather', ...), ...]"
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. ``""{featu

In [4]:
preprocessor.fit(X)
X_train = preprocessor.transform(X)
X_train

,X,Y,day,log_height,aggression_sim,play_sim,prev_tmin_c,prev_tmax_c,log_precip_mm,park_event_count,...,Approaches,Indifferent,Runs from,hl_white,hl_cinnamon,hl_gray,hl_black,has_behavior_text,tail_or_injury,mentions_white
0,0.736308,1.406853,0.524896,-0.557918,-0.663044,-0.660451,-1.099351,-1.425748,0.474788,-0.558815,...,0,1,0,1,0,0,0,0,0,0
1,-0.263873,-0.828916,-1.113383,1.816604,0.983848,1.229969,0.737320,0.087051,-0.675296,0.177742,...,0,1,0,1,0,0,0,1,0,0
2,-1.097429,-1.014988,1.929135,-0.557918,-0.663044,-0.660451,-1.431514,-0.837437,-0.675296,0.546021,...,0,0,1,0,0,0,0,0,0,0
3,-0.802523,-1.173655,-0.411264,-0.557918,-0.663044,-0.660451,1.069484,1.137606,-0.675296,1.006369,...,1,0,0,0,1,0,0,0,0,0
4,-0.682963,-1.142409,-0.879343,-0.557918,0.599566,0.274826,1.167180,1.368728,-0.675296,-1.571581,...,0,1,0,0,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2413,2.197493,1.653858,0.524896,1.816604,-0.663044,-0.660451,-1.099351,-1.425748,0.474788,0.361882,...,0,0,1,0,1,0,0,0,0,0
2414,1.603423,1.739064,0.290856,2.232406,0.983848,1.229969,-0.552257,0.087051,1.598261,0.546021,...,0,1,0,0,1,0,0,1,0,0
2415,-0.051967,0.377025,0.056816,-0.557918,-0.663044,-0.660451,1.401648,1.137606,2.174833,1.190509,...,0,1,0,0,1,0,0,0,0,0
2416,1.027106,1.615140,1.461055,-0.557918,-0.663044,-0.660451,-1.099351,-0.732382,-0.675296,2.295345,...,0,0,0,0,0,1,0,0,0,0


We save the preprocessor fitted on the training data as `preprocessor.pkl`. It includes the weather and event features so we can test them during model selection. The final `model.pkl` contains the preprocessing steps selected for Random Forest.

In [5]:
with open("preprocessor.pkl", "wb") as f:
    pickle.dump(preprocessor, f)